# 11.2 神经网络定义与训练

> **模块十一 · PyTorch深度学习实践** | 第3-5课时
>
> **学习目标：**
> 1. 掌握 `nn.Module` 自定义网络
> 2. 熟悉常用网络层（nn.Linear, nn.Conv2d, nn.ReLU 等）
> 3. 理解 nn.Sequential 快速构建网络
> 4. 掌握完整的训练循环（数据→模型→损失→优化器→训练→评估）
> 5. 掌握模型的保存与加载

---

## 1. `nn.Module` — 自定义网络

`torch.nn.Module` 是 PyTorch 中所有神经网络模块的**基类**。自定义网络需要：

1. **继承** `nn.Module`
2. 在 `__init__()` 中**定义子层**
3. 在 `forward()` 中**定义前向传播**逻辑

```python
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()      # 必须调用父类构造函数
        self.layer1 = nn.Linear(784, 256)
        self.layer2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)
        x = self.layer2(x)
        return x
```

**关键要点：**
- 不要在 `forward()` 中创建新的层（否则每次前向传播都会创建新参数）
- 只需定义 `forward()`，`backward()` 由 Autograd 自动生成

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch 版本: {torch.__version__}")

In [ ]:
# ========== 定义一个简单的 MLP (多层感知机) ==========
class SimpleMLP(nn.Module):
    """简单的多层感知机"""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # 定义网络层
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.fc1(x)           # (batch, hidden_dim)
        x = self.relu(x)         # 激活函数
        x = self.dropout(x)      # Dropout 正则化
        x = self.fc2(x)          # (batch, hidden_dim)
        x = self.relu(x)
        x = self.fc3(x)          # (batch, output_dim)
        return x

# 创建模型
model = SimpleMLP(input_dim=784, hidden_dim=128, output_dim=10)

# 查看模型结构
print(model)
print(f"\n模型参数总数: {sum(p.numel() for p in model.parameters()):,}")
print(f"可训练参数数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ========== 查看各层详细信息 ==========
print("各层参数:")
for name, param in model.named_parameters():
    print(f"  {name:20s} | 形状: {str(param.shape):20s} | 参数量: {param.numel():,}")

# ========== 模型前向传播测试 ==========
dummy_input = torch.randn(4, 784)  # batch_size=4, 输入维度=784
output = model(dummy_input)
print(f"\n输入形状: {dummy_input.shape}")
print(f"输出形状: {output.shape}")
print(f"输出示例:\n{output[0][:5].detach()}")  # 打印第一个样本的前5个输出

---
## 2. 常用网络层

### 2.1 全连接层 `nn.Linear`

全连接层执行线性变换：

$$y = xW^T + b$$

其中 $W \in \mathbb{R}^{\text{out} \times \text{in}}$，$b \in \mathbb{R}^{\text{out}}$。

In [ ]:
linear = nn.Linear(in_features=4, out_features=3)

x = torch.randn(2, 4)  # (batch=2, features=4)
y = linear(x)

print(f"输入: {x.shape} → 输出: {y.shape}")
print(f"权重 W 形状: {linear.weight.shape}")  # (out, in) = (3, 4)
print(f"偏置 b 形状: {linear.bias.shape}")    # (out,) = (3,)

# 手动验证
y_manual = x @ linear.weight.T + linear.bias
print(f"\n手动计算与 linear 输出一致: {torch.allclose(y, y_manual)}")

### 2.2 卷积层 `nn.Conv2d`

二维卷积层是处理图像数据的核心层。卷积运算：

$$y[n, c_{\text{out}}, j] = b[c_{\text{out}}] + \sum_{k=0}^{C_{\text{in}}-1} \sum_{i=0}^{c_{\text{out}}-1} W[c_{\text{out}}, k, i] * x[n, k, j + i]$$

输出尺寸计算公式：

$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2 \times \text{padding} - \text{dilation} \times (\text{kernel\_size} - 1) - 1}{\text{stride}} + 1 \right\rfloor$$

In [ ]:
conv = nn.Conv2d(
    in_channels=1,       # 输入通道数 (灰度图=1, RGB=3)
    out_channels=16,     # 输出通道数 (卷积核数量)
    kernel_size=3,       # 卷积核大小 3x3
    stride=1,            # 步长
    padding=1            # 填充 (保持尺寸不变)
)

# 单张灰度图像 28x28
x = torch.randn(2, 1, 28, 28)  # (batch=2, channels=1, H=28, W=28)
y = conv(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {y.shape}")
print(f"卷积核权重形状: {conv.weight.shape}")  # (out_ch, in_ch, kH, kW)
print(f"偏置形状: {conv.bias.shape}")          # (out_ch,)

# 不同参数的效果
conv_no_pad = nn.Conv2d(1, 16, 3, stride=1, padding=0)
conv_stride2 = nn.Conv2d(1, 16, 3, stride=2, padding=1)

y_no_pad = conv_no_pad(x)
y_stride2 = conv_stride2(x)

print(f"\n无padding:    {x.shape} → {y_no_pad.shape}")   # 28-3+1=26
print(f"stride=2:      {x.shape} → {y_stride2.shape}")   # (28+2-3)/2+1=14

### 2.3 池化层 `nn.Pool2d`

池化层用于降低特征图的空间尺寸，减少参数量：

In [ ]:
# 最大池化
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
x = torch.randn(1, 16, 28, 28)
y = maxpool(x)
print(f"MaxPool2d(2,2): {x.shape} → {y.shape}")

# 平均池化
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
y = avgpool(x)
print(f"AvgPool2d(2,2): {x.shape} → {y.shape}")

# 全局平均池化
gap = nn.AdaptiveAvgPool2d(1)  # 输出 1x1
y = gap(x)
print(f"GlobalAvgPool:  {x.shape} → {y.shape}")

### 2.4 激活函数

激活函数引入非线性，使网络能够拟合复杂函数。

| 激活函数 | 公式 | 特点 |
|----------|------|------|
| ReLU | $\max(0, x)$ | 计算快，缓解梯度消失 |
| Sigmoid | $\frac{1}{1+e^{-x}}$ | 输出(0,1)，梯度消失问题 |
| Tanh | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 输出(-1,1)，零中心 |
| LeakyReLU | $\max(0.01x, x)$ | ReLU改进，避免"死亡神经元" |
| GELU | $x\Phi(x)$ | Transformer常用 |

In [ ]:
x = torch.linspace(-4, 4, 100)

activations = {
    'ReLU': nn.ReLU()(x),
    'Sigmoid': torch.sigmoid(x),
    'Tanh': torch.tanh(x),
    'LeakyReLU': nn.LeakyReLU(0.1)(x),
    'GELU': nn.GELU()(x),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, y) in zip(axes, activations.items()):
    ax.plot(x.numpy(), y.numpy(), linewidth=2)
    ax.set_title(name, fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)

plt.suptitle('常用激活函数对比', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. `nn.Sequential` — 快速构建网络

`nn.Sequential` 是一个**有序容器**，将各层按顺序串联。适用于结构简单的网络。

In [ ]:
# ========== 使用 nn.Sequential ==========
sequential_model = nn.Sequential(
    nn.Flatten(),                    # (batch, 784)
    nn.Linear(784, 256),             # (batch, 256)
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 128),             # (batch, 128)
    nn.ReLU(),
    nn.Linear(128, 10)               # (batch, 10)
)

print(sequential_model)

dummy = torch.randn(4, 1, 28, 28)
out = sequential_model(dummy)
print(f"\n输入: {dummy.shape} → 输出: {out.shape}")

In [ ]:
# ========== Sequential + OrderedDict (命名各层) ==========
from collections import OrderedDict

named_model = nn.Sequential(OrderedDict([
    ('flatten', nn.Flatten()),
    ('fc1', nn.Linear(784, 256)),
    ('relu1', nn.ReLU()),
    ('fc2', nn.Linear(256, 10))
]))

print(named_model)
print(f"\n访问指定层: fc1 权重形状 = {named_model.fc1.weight.shape}")

### `nn.Module` vs `nn.Sequential` 对比

| 特性 | `nn.Module` | `nn.Sequential` |
|------|-------------|------------------|
| 灵活性 | ✅ 高（自定义forward） | ❌ 低（只能顺序） |
| 跳跃连接 | ✅ 支持 | ❌ 不支持 |
| 多输入/输出 | ✅ 支持 | ❌ 不支持 |
| 代码简洁度 | 一般 | ✅ 简洁 |
| 适用场景 | 复杂网络 | 简单线性网络 |

---
## 4. 损失函数 (Loss Functions)

损失函数衡量模型预测值与真实值的差距，是优化目标。

### 4.1 交叉熵损失 `nn.CrossEntropyLoss`

用于多分类任务（**内部自带 Softmax**，不要在模型末尾加 Softmax）。

$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

其中 $y_c$ 是 one-hot 编码的真实标签，$\hat{y}_c$ 是 Softmax 后的预测概率。

### 4.2 均方误差 `nn.MSELoss`

用于回归任务：

$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
# CrossEntropyLoss 演示
criterion = nn.CrossEntropyLoss()

# 模拟: batch_size=3, 类别数=10
logits = torch.randn(3, 10)          # 模型原始输出 (未经过 softmax)
targets = torch.tensor([2, 5, 0])   # 真实类别标签

loss = criterion(logits, targets)
print(f"logits 形状: {logits.shape}")
print(f"targets: {targets}")
print(f"CrossEntropy Loss: {loss.item():.4f}")

# 等价手动计算:
log_softmax = torch.log_softmax(logits, dim=1)
loss_manual = -log_softmax[0, 2] - log_softmax[1, 5] - log_softmax[2, 0]
loss_manual /= 3
print(f"手动计算 Loss: {loss_manual.item():.4f}")

print(f"\n⚠️ 注意: CrossEntropyLoss 内部已包含 Softmax!")
print(f"不要这样做: loss = CrossEntropyLoss(softmax(logits), targets)  ← 错误!")

---
## 5. 优化器 (Optimizers)

优化器根据梯度更新模型参数。PyTorch 提供了多种优化器：

| 优化器 | 更新规则 | 特点 |
|--------|----------|------|
| SGD | $\theta = \theta - \eta \nabla L$ | 最基本，可能需要动量 |
| SGD+Momentum | 引入动量项 | 加速收敛，减少震荡 |
| Adam | 自适应学习率 | 大多数场景首选 |
| RMSprop | 自适应学习率 | RNN 常用 |

SGD with Momentum：

$$v_t = \gamma v_{t-1} + \eta \nabla L(\theta)$$
$$\theta = \theta - v_t$$

Adam：

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\theta = \theta - \frac{\eta}{\sqrt{\hat{v}_t}+\epsilon}\hat{m}_t$$

In [ ]:
# 定义模型和优化器
model = SimpleMLP(784, 128, 10)

# SGD with Momentum
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam (推荐)
optimizer_adam = optim.Adam(model.parameters(), lr=0.001)

# AdamW (带权重衰减的 Adam)
optimizer_adamw = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print("优化器创建成功!")
print(f"SGD 参数组数: {len(optimizer_sgd.param_groups)}")
print(f"Adam 学习率: {optimizer_adam.param_groups[0]['lr']}")

---
## 6. 完整训练循环：MNIST 手写数字分类

下面用一个完整的例子演示 PyTorch 的训练流程：

**流程图：**

```
数据准备 → 模型定义 → 损失函数 + 优化器 → 训练循环 → 评估 → 保存模型
```

### 6.1 数据准备

使用 MNIST 手写数字数据集（28×28 灰度图，0-9 共10类）。
这里我们用 `torchvision.datasets.MNIST` 自动下载和加载。

In [ ]:
from torchvision import datasets, transforms

# ========== 数据预处理 ==========
transform = transforms.Compose([
    transforms.ToTensor(),                      # 转为 Tensor, 自动归一化到 [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST 均值和标准差
])

# ========== 下载/加载数据 ==========
train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print(f"图像形状: {train_dataset[0][0].shape}")  # (1, 28, 28)
print(f"标签类型: {type(train_dataset[0][1])}")   # int
print(f"标签范围: 0 - 9")

In [ ]:
# ========== 可视化数据集样本 ==========
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze().numpy(), cmap='gray')
    ax.set_title(f'标签: {label}', fontsize=12)
    ax.axis('off')

plt.suptitle('MNIST 数据集样本', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ========== DataLoader: 批量加载 ==========
batch_size = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,      # 训练集打乱
    num_workers=0       # Windows下设为0, Linux下可设为2-4
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False      # 测试集不打乱
)

# 查看一个 batch
images, labels = next(iter(train_loader))
print(f"一个 batch 的图像: {images.shape}")    # (64, 1, 28, 28)
print(f"一个 batch 的标签: {labels.shape}")    # (64,)
print(f"标签内容: {labels[:10]}")

### 6.2 定义 CNN 模型

这里我们使用一个简单的 CNN 来进行 MNIST 分类。

In [ ]:
class MNISTNet(nn.Module):
    """用于 MNIST 分类的简单 CNN"""
    def __init__(self):
        super().__init__()
        # 卷积部分
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # → 32×28×28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # → 64×28×28
        self.pool = nn.MaxPool2d(2, 2)                            # → 64×14×14
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)# → 128×14×14
        
        # 全连接部分
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 10)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        # x: (batch, 1, 28, 28)
        x = self.relu(self.conv1(x))   # (batch, 32, 28, 28)
        x = self.pool(x)               # (batch, 32, 14, 14)
        x = self.relu(self.conv2(x))   # (batch, 64, 14, 14)
        x = self.pool(x)               # (batch, 64, 7, 7)
        x = self.relu(self.conv3(x))   # (batch, 128, 7, 7)
        
        # 展平
        x = x.view(x.size(0), -1)       # (batch, 128*7*7)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))     # (batch, 256)
        x = self.dropout(x)
        x = self.fc2(x)                # (batch, 10)
        return x

# 创建模型
model = MNISTNet()
print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")

### 6.3 训练配置

In [ ]:
# ========== 超参数 ==========
learning_rate = 0.001
num_epochs = 10

# ========== 损失函数 ==========
criterion = nn.CrossEntropyLoss()

# ========== 优化器 ==========
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"训练配置:")
print(f"  学习率: {learning_rate}")
print(f"  训练轮数: {num_epochs}")
print(f"  批大小: {batch_size}")
print(f"  优化器: Adam")
print(f"  损失函数: CrossEntropyLoss")

### 6.4 训练循环

PyTorch 训练循环的标准模板：

```python
for epoch in range(num_epochs):
    model.train()                    # 设为训练模式
    for batch in train_loader:
        # 1. 前向传播
        output = model(inputs)
        loss = criterion(output, targets)
        
        # 2. 反向传播
        optimizer.zero_grad()        # 清零梯度!
        loss.backward()                # 计算梯度
        
        # 3. 更新参数
        optimizer.step()              # 更新权重
```

In [ ]:
# ========== 训练循环 ==========
train_losses = []
train_accs = []
test_losses = []
test_accs = []

for epoch in range(num_epochs):
    # ---- 训练阶段 ----
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        optimizer.zero_grad()   # 清零梯度
        loss.backward()         # 计算梯度
        optimizer.step()        # 更新参数
        
        # 统计
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    # ---- 验证阶段 ----
    model.eval()  # 设为评估模式 (关闭 Dropout, BN 使用 running stats)
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # 不计算梯度
        for images, labels in test_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    test_loss = test_loss / total
    test_acc = 100.0 * correct / total
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f"Epoch [{epoch+1:2d}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

In [ ]:
# ========== 训练曲线可视化 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
axes[0].plot(range(1, num_epochs+1), train_losses, 'b-o', label='训练损失', markersize=4)
axes[0].plot(range(1, num_epochs+1), test_losses, 'r-o', label='测试损失', markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('损失曲线', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy 曲线
axes[1].plot(range(1, num_epochs+1), train_accs, 'b-o', label='训练准确率', markersize=4)
axes[1].plot(range(1, num_epochs+1), test_accs, 'r-o', label='测试准确率', markersize=4)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('准确率曲线', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.5 模型预测可视化

In [ ]:
# ========== 可视化预测结果 ==========
model.eval()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        img, true_label = test_dataset[i]
        output = model(img.unsqueeze(0))  # 添加 batch 维度
        pred_label = output.argmax(dim=1).item()
        prob = torch.softmax(output, dim=1)[0, pred_label].item()
        
        ax.imshow(img.squeeze().numpy(), cmap='gray')
        color = 'green' if pred_label == true_label else 'red'
        ax.set_title(f'真:{true_label} 预:{pred_label} ({prob:.1%})', 
                     color=color, fontsize=11)
        ax.axis('off')

plt.suptitle('模型预测结果 (绿色=正确, 红色=错误)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. 模型保存与加载

### 7.1 保存方式

| 方式 | API | 说明 |
|------|-----|------|
| 保存整个模型 | `torch.save(model, path)` | 保存模型结构+参数（依赖代码） |
| 保存参数字典 | `torch.save(model.state_dict(), path)` | ✅ 推荐，只保存权重 |

In [ ]:
import os

save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

# ========== 方式1: 保存 state_dict (推荐) ==========
save_path = os.path.join(save_dir, 'mnist_cnn.pth')
torch.save(model.state_dict(), save_path)
print(f"模型参数已保存到: {save_path}")
print(f"文件大小: {os.path.getsize(save_path) / 1024:.1f} KB")

# ========== 方式2: 保存完整模型 ==========
save_path_full = os.path.join(save_dir, 'mnist_cnn_full.pth')
torch.save(model, save_path_full)
print(f"完整模型已保存到: {save_path_full}")

# ========== 方式3: 保存训练状态 (checkpoint) ==========
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'test_acc': max(test_accs)
}
ckpt_path = os.path.join(save_dir, 'mnist_checkpoint.pth')
torch.save(checkpoint, ckpt_path)
print(f"Checkpoint 已保存到: {ckpt_path}")

In [ ]:
# ========== 加载模型 ==========

# 方式1: 加载 state_dict (推荐)
new_model = MNISTNet()                       # 先创建模型
new_model.load_state_dict(torch.load(save_path, weights_only=True))  # 加载参数
new_model.eval()
print("✅ state_dict 加载成功!")

# 验证加载后的模型
with torch.no_grad():
    test_img, test_label = test_dataset[0]
    pred = new_model(test_img.unsqueeze(0)).argmax(1).item()
    print(f"测试样本预测: {pred}, 真实标签: {test_label}")

# 方式3: 加载 checkpoint
checkpoint = torch.load(ckpt_path, weights_only=False)
print(f"\nCheckpoint 信息:")
print(f"  训练轮数: {checkpoint['epoch']}")
print(f"  最高测试准确率: {checkpoint['test_acc']:.2f}%")

### 7.2 `model.train()` vs `model.eval()`

| 模式 | Dropout | BatchNorm |
|------|---------|-----------|
| `model.train()` | ✅ 启用（随机丢弃） | 使用当前 batch 统计量 |
| `model.eval()` | ❌ 禁用（全部保留） | 使用 running 统计量 |

**务必记住：**
- 训练时调用 `model.train()`
- 评估/测试时调用 `model.eval()`

---
## 8. 本节小结

### PyTorch 训练完整流程

```
1. 数据准备
   ├── transforms.Compose()      # 数据增强/预处理
   ├── Dataset                   # 数据集
   └── DataLoader                # 批量加载器

2. 模型定义
   ├── class MyNet(nn.Module)    # 自定义网络
   ├── __init__()                # 定义层
   └── forward()                 # 前向传播

3. 训练配置
   ├── criterion = LossFn()      # 损失函数
   └── optimizer = Optimizer()   # 优化器

4. 训练循环
   ├── optimizer.zero_grad()     # 清零梯度
   ├── loss = criterion(output, target)  # 计算损失
   ├── loss.backward()           # 反向传播
   └── optimizer.step()          # 更新参数

5. 评估与保存
   ├── model.eval() + no_grad()  # 评估
   └── torch.save(state_dict())  # 保存
```

---

## 📝 练习题

### 练习1: 自定义网络

定义一个 CNN，用于 CIFAR-10 分类（32×32 RGB 图像，10 类），要求：
- 至少 2 个卷积层 + 池化层
- 至少 2 个全连接层
- 使用 BatchNorm 和 Dropout
- 打印模型参数总数

**提示：** CIFAR-10 图像尺寸为 `3×32×32`

### 练习2: 训练循环

使用上面定义的网络和 MNIST 数据集完成训练：
- 使用 SGD + Momentum 优化器
- 学习率从 0.1 开始，每 5 个 epoch 乘以 0.1
- 训练 15 个 epoch
- 记录并绘制训练/测试 loss 曲线

**提示：** 使用 `optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)`

### 练习3: 对比实验

对比以下设置对 MNIST 测试准确率的影响：
1. 使用 ReLU vs Sigmoid 激活函数
2. 有 Dropout vs 无 Dropout
3. SGD vs Adam 优化器

用表格记录结果并分析原因。